In [1]:
import pickle
import numpy as np
import torch

file_path = '../Literature/DFOL/test_dir/'

# read the x_train.dt and y_train.dt files
with open(file_path + 'x_train.dt', 'rb') as xf:
    x_train = pickle.load(xf)
    xf.close()
with open(file_path + 'y_train.dt', 'rb') as yf:
    y_train = pickle.load(yf)
    yf.close()

print("👉 The x_train data is:")
print(len(x_train[0]))
print("👉 The y_train data is:")
print(len(y_train))

x_train_np = np.concatenate(x_train, axis=1)
X_train = torch.tensor(x_train_np[:, :-1], dtype=torch.float32)
y_train = torch.tensor(x_train_np[:, -1], dtype=torch.float32) 

# Load the predicate list
with open(file_path + 'predicate.list', 'rb') as f:
    predicate_list = pickle.load(f)
    f.close()

# Load relation variable permutations
with open(file_path + 'relation_variable.dt', 'rb') as f:
    relation_variable_permutations = pickle.load(f)
    f.close()

# Load relation names
with open(file_path + 'all_relation_dic.dt', 'rb') as f:
    all_relations = pickle.load(f)
    f.close()

print("Predicates:", len(predicate_list))
print("Relations:", len(all_relations))
print("Variable permutations:", len(relation_variable_permutations))

👉 The x_train data is:
38416
👉 The y_train data is:
38416
Predicates: 660
Relations: 55
Variable permutations: 55


In [2]:
def get_predicate_name(index, predicate_list, relation_variable_permutations, all_relations):
    """
    Convert a feature index in X_train to its symbolic predicate name.
    """
    # Get variable mapping (e.g., (0, 1) for X, Y)
    var_tuple = predicate_list[index]
    
    # Variable name mapping
    var_names = {0: 'X1', 1: 'X2', 2: 'X3', 3: 'X4'}
    
    # Find which relation this belongs to
    cumulative_index = 0
    for relation_name, permutations in relation_variable_permutations.items():
        if cumulative_index <= index < cumulative_index + len(permutations):
            # Found the relation
            local_index = index - cumulative_index
            var_pair = permutations[local_index]
            
            # Build symbolic predicate
            symbolic = f"{relation_name}({var_names[var_pair[0]]}, {var_names[var_pair[1]]})"
            return symbolic
        
        cumulative_index += len(permutations)
    
    return "Unknown"

# Example: Map X_train column 0
body_predicates = []
for idx in range(len(predicate_list)):
    predicate_name = get_predicate_name(idx, predicate_list, 
                                        relation_variable_permutations, 
                                        all_relations)
    print(f"X_train[:, {idx}] = {predicate_name}")
    body_predicates.append(predicate_name)

X_train[:, 0] = militaryalliance(X1, X2)
X_train[:, 1] = militaryalliance(X1, X3)
X_train[:, 2] = militaryalliance(X1, X4)
X_train[:, 3] = militaryalliance(X2, X1)
X_train[:, 4] = militaryalliance(X2, X3)
X_train[:, 5] = militaryalliance(X2, X4)
X_train[:, 6] = militaryalliance(X3, X1)
X_train[:, 7] = militaryalliance(X3, X2)
X_train[:, 8] = militaryalliance(X3, X4)
X_train[:, 9] = militaryalliance(X4, X1)
X_train[:, 10] = militaryalliance(X4, X2)
X_train[:, 11] = militaryalliance(X4, X3)
X_train[:, 12] = intergovorgs3(X1, X2)
X_train[:, 13] = intergovorgs3(X1, X3)
X_train[:, 14] = intergovorgs3(X1, X4)
X_train[:, 15] = intergovorgs3(X2, X1)
X_train[:, 16] = intergovorgs3(X2, X3)
X_train[:, 17] = intergovorgs3(X2, X4)
X_train[:, 18] = intergovorgs3(X3, X1)
X_train[:, 19] = intergovorgs3(X3, X2)
X_train[:, 20] = intergovorgs3(X3, X4)
X_train[:, 21] = intergovorgs3(X4, X1)
X_train[:, 22] = intergovorgs3(X4, X2)
X_train[:, 23] = intergovorgs3(X4, X3)
X_train[:, 24] = relbooktranslations(X

In [4]:
from utils import create_V_matrix

head_predicate = body_predicates[252]  # Example head predicate
variable_matrix, variables = create_V_matrix(body_predicates)
head_vars = [0, 1]
connected_vars = [2, 3]

In [5]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy
import matplotlib.pyplot as plt
from ANDRE_Net import ANDRE
from collections import Counter
from utils import check_binary_dataset, create_V_matrix
from utils import plot_andre_weights
from utils import interpret_neuron_weights_entropy, interpret_neuron_weights
from utils import compute_acc_ratio_all_rules_attention
from utils import extract_rules
from utils import parse_rule_and_generate_data
import numpy as np

# Parameters
Num_rules = 4
num_samples = 512
num_epochs = 500
num_restarts = 3
batch_size = 128
sim_weight, ent_weight, rr_weight, conn_weight = 0.1, 1, 0.01, 2

seed = 42
visualize_weights = False
input_dim = X_train.shape[1]
structured_data = True
acc_threshold = 0.95
dataset_type = 'binary'  # 'binary' or 'continuous'
if check_binary_dataset(X_train):
    print("X_train is a binary dataset. We are perturbing the data for better training.")
    # perturb data by adding small noise to zero and reducing one to less than one
    X_train = X_train - 0.1 * torch.rand_like(X_train) * (X_train - 0.5).sign()
    # clip to [0, 1]
    X_train = torch.clamp(X_train, 0.0, 1.0)

X_train is a binary dataset. We are perturbing the data for better training.


In [ ]:
# Cosine similarity function
cosine = nn.CosineSimilarity(dim=1)

# Multi-Restart Training
best_state_dict = None
best_loss = float('inf')
best_eval_metric = -float('inf')
best_losses = []

# Move data to device
# X_train = X_train.to(device)
# y_train = y_train.to(device)

# for num_rules in range(Num_rules):
for num_rules in [3]:

    for restart in range(num_restarts):
        model = ANDRE(num_rules + 1, pot_body_pred_dim=input_dim, variable_matrix=variable_matrix if structured_data else None, lr=0.005, weight_decay=1e-4)
        model = model

        losses_andre = []

        for epoch in range(num_epochs):
            model.train()

            # Shuffle dataset
            perm = torch.randperm(num_samples)
            X_train_shuffled = X_train[perm]
            y_train_shuffled = y_train[perm]

            epoch_loss = 0.0

            # Mini-batch training loop
            for i in range(0, num_samples, batch_size):
                xb = X_train_shuffled[i:i + batch_size]
                yb = y_train_shuffled[i:i + batch_size]

                model.optimizer.zero_grad()
                outputs = model(xb).squeeze()

                # Binary Cross-Entropy Loss
                bce_loss = model.criterion(outputs, yb)

                # Similarity Penalty
                sim_loss = 0.0
                for i in range(num_rules + 1):
                    wi = model.saved_weights[i].view(input_dim, -1)
                    for j in range(i + 1, num_rules + 1):
                        wj = model.saved_weights[j].view(input_dim, -1)
                        sim_loss += nn.CosineSimilarity(dim=1)(wi, wj).mean()
                sim_loss = sim_loss / (num_rules + 1)

                if structured_data:
                    M = model.M_matrices
                    M_head = [M[i][head_vars] for i in range(num_rules + 1)]
                    head_var_probs = torch.prod(torch.stack(M_head), dim=0)

                    # Range Restriction Loss
                    rr_loss_1 = nn.MSELoss()(head_var_probs, torch.ones_like(head_var_probs))
                    rr_loss_2 = nn.MSELoss()(head_var_probs, torch.ones_like(head_var_probs))
                    rr_loss = rr_loss_1 + rr_loss_2
                    rr_loss = rr_loss.sum() / xb.size(0)

                    # Connectivity Loss
                    M_connected = [M[i][connected_vars] for i in range(num_rules + 1)]
                    Sum_Fx = sum(torch.exp(-0.5 * ((M - 1.0) / 0.2)**2) / (0.2 * math.sqrt(2 * math.pi)) for M in M_connected)
                    conn_loss_1 = 0.5 * Sum_Fx / len(M_connected)
                    conn_loss_2 = nn.MSELoss()(conn_loss_1, 2 * torch.ones_like(conn_loss_1))
                    conn_loss = 3 * conn_loss_1 + 0.01 * conn_loss_2
                    conn_loss = conn_loss.sum() / xb.size(0)

                    # Entropy Loss
                    entropy_term = sum(model.entropies)
                    entropy_term = entropy_term / xb.size(0)

                    # Total Loss
                    total_loss = bce_loss + ent_weight * entropy_term + rr_weight * rr_loss + conn_weight * conn_loss
                    # print("bce_loss", bce_loss.item(), "entropy_term:", entropy_term.item(), "rr_loss:", rr_loss.item(), 
                    #       "conn_loss:", conn_loss.item(), "penalty:", sim_loss.item())

                else:
                    entropy_term = sum(model.entropies)
                    total_loss = bce_loss + ent_weight * entropy_term

                total_loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                model.optimizer.step()

                epoch_loss += total_loss.item()

            model.scheduler.step()
            losses_andre.append(epoch_loss)
            # print(f"Num Rules: {num_rules + 1}, Restart {restart+1}/{num_restarts}, Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}.")

        # Evaluate model
        metrics = model.evaluate(X_train, y_train)
        accuracy = metrics['accuracy']
        precision = metrics['precision']
        recall = metrics['recall']
        f1_score = metrics['f1_score']

        # hard_metric = 0.25 * (accuracy + precision + recall + f1_score)
        hard_metric = accuracy * precision * recall
        soft_accuracy = accuracy

        eval_metric = soft_accuracy

        print(f"Num Rules: {num_rules + 1}, Restart {restart+1}/{num_restarts}, Evaluation Metric: {eval_metric:.4f}.")
        if best_eval_metric < eval_metric:
            print(f"**** New best model ****")
            best_eval_metric = eval_metric
            best_state_dict = deepcopy(model.state_dict())
            best_losses = losses_andre
            best_num_rules = num_rules + 1

            acc = metrics['accuracy']
            print(f"Restart {restart+1}: acc_ratio = {acc:.4f}")
            if acc > acc_threshold:
                print(f"The accuracy threshold of {acc_threshold} has been reached. Stopping further restarts.")
                break

# Load the best model
best_model = ANDRE(best_num_rules, pot_body_pred_dim=input_dim, variable_matrix=variable_matrix if structured_data else None, lr=0.005, weight_decay=1e-4)
best_model.load_state_dict(best_state_dict)

# Evaluate best model
best_model.print_evaluation(X_train, y_train)

# Plot loss curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), best_losses, label="Loss", color="blue")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title(f"Training Loss - ({best_num_rules} Rules)")
plt.legend()
plt.grid(True)
plt.show()

# Save final weights for inspection
Weights_andre = []
for rule in best_model.rules:
    rule_weights = []
    for neuron in rule:
        weights = torch.softmax(neuron.raw_weights, dim=0).detach().numpy().tolist()
        # weights = torch.sigmoid(neuron.raw_weights).detach().numpy().tolist()
        rule_weights.append(weights)
    Weights_andre.append(rule_weights)
# print("\nFinal Weights of the Network:")
# for i, weights in enumerate(Weights_andre):
#     print(f"Rule {i+1}: {weights}")

if visualize_weights:
    plot_andre_weights(Weights_andre, input_dim=input_dim, predicates=body_predicates, structured_data=structured_data)

# Rule interpretation
final_rule = extract_rules(Weights_andre=Weights_andre, X_train=X_train, 
                           dataset_type=dataset_type, body_predicates=body_predicates,
                           head_predicate=head_predicate)
print("\n======================================")
print("Final Rule:")
print(final_rule)
print("======================================")


KeyboardInterrupt: 